In [1]:
import pandas as pd
import numpy as np
import zipfile
import warnings
import xml.etree.ElementTree as ET

warnings.filterwarnings('ignore')

print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")
print("Todo listo")

pandas:  2.1.2
numpy:   1.26.1
Todo listo


## M1 — Pipeline de datos
Cargamos los tres archivos fuente y los preparamos para el análisis.

In [2]:
PIRAMIDE_PATH = r"C:\Users\jcrs2\Documents\Akademia\akademia-residencias\Datos caso práctico 2026 - piramide.ods"
RENTA_PATH    = r"C:\Users\jcrs2\Documents\Akademia\akademia-residencias\Datos caso práctico 2026 - renta y localizacion.ods"

print("Rutas definidas:")
print(f"  Pirámide: {PIRAMIDE_PATH}")
print(f"  Renta:    {RENTA_PATH}")

Rutas definidas:
  Pirámide: C:\Users\jcrs2\Documents\Akademia\akademia-residencias\Datos caso práctico 2026 - piramide.ods
  Renta:    C:\Users\jcrs2\Documents\Akademia\akademia-residencias\Datos caso práctico 2026 - renta y localizacion.ods


### Paso 1 — Leer el archivo de localización

In [4]:
df_loc = pd.read_excel(RENTA_PATH, sheet_name='localizacion', engine='odf')

print(f"Filas:    {df_loc.shape[0]}")
print(f"Columnas: {df_loc.shape[1]}")
print(f"Nombres de columnas: {df_loc.columns.tolist()}")

df_loc.head()

Filas:    36333
Columnas: 5
Nombres de columnas: ['CPRO', 'CMUN', 'Seccion', 'longitud', 'latitud']


,CPRO,CMUN,Seccion,longitud,latitud
0,1.0,1.0,100101001,-2.521144,42.813781
1,NaN,NaN,100101002,-2.499572,42.847201
2,NaN,2.0,100201001,-2.996599,42.987151
3,NaN,NaN,100201002,-2.948063,43.034617
4,NaN,NaN,100201003,-3.007986,43.057505


### Paso 2 — Explorar localización en profundidad

In [5]:
# .info() → tipos de dato y cuántos valores no-nulos hay por columna
print("=== INFO ===")
df_loc.info()

# .isnull().sum() → conteo exacto de nulos por columna
print("\n=== NULOS POR COLUMNA ===")
print(df_loc.isnull().sum())

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36333 entries, 0 to 36332
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   CPRO      52 non-null     float64
 1   CMUN      8131 non-null   float64
 2   Seccion   36333 non-null  int64  
 3   longitud  36333 non-null  float64
 4   latitud   36333 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 1.4 MB

=== NULOS POR COLUMNA ===
CPRO        36281
CMUN        28202
Seccion         0
longitud        0
latitud         0
dtype: int64


### Paso 3 — Rellenar valores vacíos en CPRO y CMUN

In [6]:
df_loc['CPRO'] = df_loc['CPRO'].ffill()
df_loc['CMUN'] = df_loc['CMUN'].ffill()

# Convertir a entero ahora que no hay nulos
df_loc['CPRO'] = df_loc['CPRO'].astype(int)
df_loc['CMUN'] = df_loc['CMUN'].astype(int)

print("=== NULOS DESPUÉS DE FFILL ===")
print(df_loc.isnull().sum())

print("\n=== PRIMERAS FILAS ===")
df_loc.head(8)

=== NULOS DESPUÉS DE FFILL ===
CPRO        0
CMUN        0
Seccion     0
longitud    0
latitud     0
dtype: int64

=== PRIMERAS FILAS ===


,CPRO,CMUN,Seccion,longitud,latitud
0,1,1,100101001,-2.521144,42.813781
1,1,1,100101002,-2.499572,42.847201
2,1,2,100201001,-2.996599,42.987151
3,1,2,100201002,-2.948063,43.034617
4,1,2,100201003,-3.007986,43.057505
5,1,2,100201004,-3.002510,43.060785
6,1,2,100201005,-2.998707,43.054978
7,1,2,100201006,-2.965457,43.065538


### Paso 4 — Leer la hoja de municipios

In [7]:
df_muni = pd.read_excel(RENTA_PATH, sheet_name='situacion', engine='odf', header=1)

print(f"Filas:    {df_muni.shape[0]}")
print(f"Columnas: {df_muni.columns.tolist()}")

df_muni.head()

Filas:    8124
Columnas: ['CODAUTO', 'CPRO', 'CMUN', 'DC', 'NOMBRE']


,CODAUTO,CPRO,CMUN,DC,NOMBRE
0,16,1,51,3,Agurain/Salvatierra
1,16,1,1,4,Alegría-Dulantzi
2,16,1,2,9,Amurrio
3,16,1,49,3,Añana
4,16,1,3,5,Aramaio


### Paso 5 — Leer la hoja de renta

In [8]:
df_renta_raw = pd.read_excel(RENTA_PATH, sheet_name='renta', engine='odf', header=None)

print(f"Tamaño: {df_renta_raw.shape}")
df_renta_raw.head(12)

Tamaño: (55103, 5)


,0,1,2,3,4
0,Tablas nacionales,NaN,NaN,NaN,NaN
1,Total Nacional,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,Indicadores de renta media,NaN,NaN,NaN,NaN
4,Unidades: Euros,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN
6,,Renta media por persona,NaN,Renta media por hogar,NaN
7,,2024,2023,2024,2023
8,01001 Alegría-Dulantzi,13086,12936,34373,33702
9,0100101 Alegría-Dulantzi distrito 01,,,,


### Paso 6 — Limpiar la hoja de renta

In [9]:
# Cogemos solo las columnas que nos interesan: nombre (0), renta_persona_2024 (1), renta_hogar_2024 (3)
df_renta = df_renta_raw.iloc[8:, [0, 1, 3]].copy()

df_renta.columns = ['NOMBRE', 'renta_persona_2024', 'renta_hogar_2024']

# Convertir a numérico — las filas de distrito y sección quedarán como NaN
df_renta['renta_hogar_2024']   = pd.to_numeric(df_renta['renta_hogar_2024'],   errors='coerce')
df_renta['renta_persona_2024'] = pd.to_numeric(df_renta['renta_persona_2024'], errors='coerce')

# Quedarnos solo con filas que tienen renta — eso filtra distritos y secciones vacíos
df_renta = df_renta.dropna(subset=['renta_hogar_2024']).reset_index(drop=True)

print(f"Municipios con datos de renta: {df_renta.shape[0]}")
df_renta.head(8)

Municipios con datos de renta: 50345


,NOMBRE,renta_persona_2024,renta_hogar_2024
0,01001 Alegría-Dulantzi,13086.0,34373.0
1,01002 Amurrio,13691.0,33936.0
2,01003 Aramaio,17194.0,46567.0
3,01004 Artziniega,13185.0,32680.0
4,01006 Armiñón,11330.0,28266.0
5,01008 Arratzua-Ubarrundia,18089.0,47872.0
6,01009 Asparrena,14134.0,31889.0
7,01010 Ayala/Aiara,13635.0,36970.0


### Paso 7 — Entender qué hay en los datos de renta

In [10]:
# Los primeros caracteres del nombre nos dicen el nivel geográfico
# Municipio:       "01001 Alegría..."      → código de 5 dígitos
# Comunidad aut.:  "16 País Vasco..."      → código de 2 dígitos
# Total nacional:  "Total Nacional"

# Miramos cuántos caracteres tienen los códigos al principio
df_renta['codigo_str'] = df_renta['NOMBRE'].astype(str).str.extract(r'^(\d+)')
df_renta['longitud_codigo'] = df_renta['codigo_str'].str.len()

print("Distribución por longitud de código:")
print(df_renta['longitud_codigo'].value_counts().sort_index())

df_renta.head(10)

Distribución por longitud de código:
longitud_codigo
5      6811
7      9125
10    34409
Name: count, dtype: int64


,NOMBRE,renta_persona_2024,renta_hogar_2024,codigo_str,longitud_codigo
0,01001 Alegría-Dulantzi,13086.0,34373.0,01001,5
1,01002 Amurrio,13691.0,33936.0,01002,5
2,01003 Aramaio,17194.0,46567.0,01003,5
3,01004 Artziniega,13185.0,32680.0,01004,5
4,01006 Armiñón,11330.0,28266.0,01006,5
5,01008 Arratzua-Ubarrundia,18089.0,47872.0,01008,5
6,01009 Asparrena,14134.0,31889.0,01009,5
7,01010 Ayala/Aiara,13635.0,36970.0,01010,5
8,01011 Baños de Ebro/Mañueta,11794.0,29932.0,01011,5
9,01013 Barrundia,15419.0,39003.0,01013,5


### Paso 8 — Filtrar solo municipios

In [11]:
# Filtrar solo municipios — código de 5 dígitos
df_renta = df_renta[df_renta['longitud_codigo'] == 5].copy()

# Extraer CPRO (primeros 2 dígitos) y CMUN (últimos 3 dígitos)
df_renta['CPRO'] = df_renta['codigo_str'].str[:2].astype(int)
df_renta['CMUN'] = df_renta['codigo_str'].str[2:].astype(int)

# Limpiar columnas auxiliares que ya no necesitamos
df_renta = df_renta.drop(columns=['codigo_str', 'longitud_codigo'])

df_renta = df_renta.reset_index(drop=True)

print(f"Municipios con renta: {df_renta.shape[0]}")
print(f"Columnas: {df_renta.columns.tolist()}")
df_renta.head(8)

Municipios con renta: 6811
Columnas: ['NOMBRE', 'renta_persona_2024', 'renta_hogar_2024', 'CPRO', 'CMUN']


,NOMBRE,renta_persona_2024,renta_hogar_2024,CPRO,CMUN
0,01001 Alegría-Dulantzi,13086.0,34373.0,1,1
1,01002 Amurrio,13691.0,33936.0,1,2
2,01003 Aramaio,17194.0,46567.0,1,3
3,01004 Artziniega,13185.0,32680.0,1,4
4,01006 Armiñón,11330.0,28266.0,1,6
5,01008 Arratzua-Ubarrundia,18089.0,47872.0,1,8
6,01009 Asparrena,14134.0,31889.0,1,9
7,01010 Ayala/Aiara,13635.0,36970.0,1,10


### Paso 9 — Leer la pirámide poblacional
El archivo ODS pesa 106MB comprimido pero su XML interno ocupa 1.3GB.
Pandas no puede cargarlo entero en memoria — lo procesamos fila a fila.

In [12]:
def leer_piramide_ods(filepath, max_cols=70):
    NS_TABLE = 'urn:oasis:names:tc:opendocument:xmlns:table:1.0'
    NS_TEXT  = 'urn:oasis:names:tc:opendocument:xmlns:text:1.0'
    
    filas = []
    
    with zipfile.ZipFile(filepath) as z:
        with z.open('content.xml') as f:
            for event, elem in ET.iterparse(f, events=['end']):
                
                if elem.tag == f'{{{NS_TABLE}}}table-row':
                    fila = []
                    
                    for cell in elem:
                        if len(fila) >= max_cols:
                            break
                        
                        repeat = int(cell.get(
                            f'{{{NS_TABLE}}}number-columns-repeated', 1))
                        repeat = min(repeat, max_cols - len(fila))
                        
                        textos = cell.findall(f'.//{{{NS_TEXT}}}p')
                        valor  = textos[0].text if textos else None
                        
                        fila.extend([valor] * repeat)
                    
                    if any(v is not None for v in fila):
                        filas.append(fila[:max_cols])
                    
                    elem.clear()
    
    return filas

print("Función definida correctamente")

Función definida correctamente


### Paso 10 — Ejecutar la lectura de la pirámide
Esta celda tarda 2-3 minutos. Es normal.

In [13]:
print("Leyendo pirámide poblacional... no toques nada hasta que termine")

filas_raw = leer_piramide_ods(PIRAMIDE_PATH)

print(f"\nLectura completada")
print(f"Filas leídas: {len(filas_raw)}")
print(f"\nEjemplo fila 0: {filas_raw[0]}")
print(f"Ejemplo fila 1: {filas_raw[1][:8]}")
print(f"Ejemplo fila 2: {filas_raw[2][:5]}")
print(f"Ejemplo fila 3: {filas_raw[3][:5]}")

Leyendo pirámide poblacional... no toques nada hasta que termine

Lectura completada
Filas leídas: 145370

Ejemplo fila 0: [None, 'Ambos sexos.', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'Hombres', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'Mujeres', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
Ejemplo fila 1: ['Seccion Censal', 'Total', '(0,4)', '(5,9)', '(10,14)', '(15,19)', '(20,24)', '(25,29)']
Ejemplo fila 2: ['0000000000', '47385107', '1887689', '2275778', '2526140']
Ejemplo fila 3: ['0100101001', '1382', '43', '78', '123']


### Paso 11 — Construir el DataFrame de la pirámide

In [14]:
grupos_edad = ['0_4','5_9','10_14','15_19','20_24','25_29','30_34','35_39',
               '40_44','45_49','50_54','55_59','60_64','65_69','70_74',
               '75_79','80_84','85_89','90_94','95_99','100mas']

columnas = (
    ['seccion_censal'] +
    ['total_ambos'] + [f'ambos_{g}' for g in grupos_edad] +
    ['total_hombres'] + [f'hombres_{g}' for g in grupos_edad] +
    ['total_mujeres'] + [f'mujeres_{g}' for g in grupos_edad]
)

print(f"Columnas definidas: {len(columnas)}")
print(f"Ejemplo: {columnas[:5]}")
print(f"...{columnas[22:27]}")
print(f"...{columnas[44:49]}")

Columnas definidas: 67
Ejemplo: ['seccion_censal', 'total_ambos', 'ambos_0_4', 'ambos_5_9', 'ambos_10_14']
...['ambos_100mas', 'total_hombres', 'hombres_0_4', 'hombres_5_9', 'hombres_10_14']
...['hombres_100mas', 'total_mujeres', 'mujeres_0_4', 'mujeres_5_9', 'mujeres_10_14']


### Paso 12 — Crear el DataFrame y convertir a numérico

In [19]:
columnas = (
    ['seccion_censal'] +
    ['total_ambos'] + [f'ambos_{g}' for g in grupos_edad] +
    ['total_hombres'] + [f'hombres_{g}' for g in grupos_edad] +
    ['total_mujeres'] + [f'mujeres_{g}' for g in grupos_edad] +
    ['basura_1', 'basura_2', 'basura_3']
)

df_pir = pd.DataFrame(filas_raw[3:], columns=columnas)

# Eliminar las columnas basura
df_pir = df_pir.drop(columns=['basura_1', 'basura_2', 'basura_3'])

# Convertir todas las columnas excepto seccion_censal a numérico
cols_numericas = [c for c in df_pir.columns if c != 'seccion_censal']
df_pir[cols_numericas] = df_pir[cols_numericas].apply(
    pd.to_numeric, errors='coerce'
)

print(f"Secciones censales: {df_pir.shape[0]:,}")
print(f"Columnas:           {df_pir.shape[1]}")
print(f"\nNulos por columna (primeras 5):")
print(df_pir.isnull().sum().head())
print(f"\nEjemplo:")
df_pir.head(3)

Secciones censales: 145,367
Columnas:           67

Nulos por columna (primeras 5):
seccion_censal        26
total_ambos       109033
ambos_0_4         109033
ambos_5_9         109033
ambos_10_14       109033
dtype: int64

Ejemplo:


,seccion_censal,total_ambos,ambos_0_4,ambos_5_9,ambos_10_14,ambos_15_19,ambos_20_24,ambos_25_29,ambos_30_34,ambos_35_39,...,mujeres_55_59,mujeres_60_64,mujeres_65_69,mujeres_70_74,mujeres_75_79,mujeres_80_84,mujeres_85_89,mujeres_90_94,mujeres_95_99,mujeres_100mas
0,0100101001,1382.0,43.0,78.0,123.0,82.0,65.0,54.0,58.0,77.0,...,47.0,29.0,33.0,17.0,20.0,8.0,9.0,4.0,0.0,0.0
1,0100101002,1543.0,73.0,99.0,137.0,103.0,71.0,58.0,67.0,90.0,...,46.0,26.0,30.0,22.0,18.0,10.0,13.0,6.0,3.0,0.0
2,0100201001,1888.0,73.0,100.0,85.0,77.0,73.0,81.0,92.0,107.0,...,81.0,96.0,73.0,65.0,31.0,37.0,24.0,12.0,7.0,0.0


In [20]:
# Eliminar filas sin seccion_censal y sin datos de población
df_pir = df_pir.dropna(subset=['seccion_censal', 'total_ambos'])
df_pir = df_pir.reset_index(drop=True)

print(f"Secciones censales tras limpieza: {df_pir.shape[0]:,}")
print(f"\nNulos por columna (primeras 5):")
print(df_pir.isnull().sum().head())
print(f"\nRango de total_ambos:")
print(f"  Mínimo: {df_pir['total_ambos'].min():,.0f}")
print(f"  Máximo: {df_pir['total_ambos'].max():,.0f}")
print(f"  Media:  {df_pir['total_ambos'].mean():,.0f}")
df_pir.head(3)

Secciones censales tras limpieza: 36,334

Nulos por columna (primeras 5):
seccion_censal    0
total_ambos       0
ambos_0_4         0
ambos_5_9         0
ambos_10_14       0
dtype: int64

Rango de total_ambos:
  Mínimo: 3
  Máximo: 16,492
  Media:  1,305


,seccion_censal,total_ambos,ambos_0_4,ambos_5_9,ambos_10_14,ambos_15_19,ambos_20_24,ambos_25_29,ambos_30_34,ambos_35_39,...,mujeres_55_59,mujeres_60_64,mujeres_65_69,mujeres_70_74,mujeres_75_79,mujeres_80_84,mujeres_85_89,mujeres_90_94,mujeres_95_99,mujeres_100mas
0,0100101001,1382.0,43.0,78.0,123.0,82.0,65.0,54.0,58.0,77.0,...,47.0,29.0,33.0,17.0,20.0,8.0,9.0,4.0,0.0,0.0
1,0100101002,1543.0,73.0,99.0,137.0,103.0,71.0,58.0,67.0,90.0,...,46.0,26.0,30.0,22.0,18.0,10.0,13.0,6.0,3.0,0.0
2,0100201001,1888.0,73.0,100.0,85.0,77.0,73.0,81.0,92.0,107.0,...,81.0,96.0,73.0,65.0,31.0,37.0,24.0,12.0,7.0,0.0


### Paso 13 — Guardar los datasets limpios
A partir de aquí siempre leeremos los CSV, nunca los ODS originales.

In [21]:
df_pir.to_csv('piramide_limpia.csv', index=False)
df_loc.to_csv('localizacion_limpia.csv', index=False)
df_muni.to_csv('municipios_limpios.csv', index=False)
df_renta.to_csv('renta_limpia.csv', index=False)

print("Archivos guardados:")
print(f"  piramide_limpia.csv     → {df_pir.shape[0]:,} filas, {df_pir.shape[1]} columnas")
print(f"  localizacion_limpia.csv → {df_loc.shape[0]:,} filas, {df_loc.shape[1]} columnas")
print(f"  municipios_limpios.csv  → {df_muni.shape[0]:,} filas, {df_muni.shape[1]} columnas")
print(f"  renta_limpia.csv        → {df_renta.shape[0]:,} filas, {df_renta.shape[1]} columnas")

Archivos guardados:
  piramide_limpia.csv     → 36,334 filas, 67 columnas
  localizacion_limpia.csv → 36,333 filas, 5 columnas
  municipios_limpios.csv  → 8,124 filas, 5 columnas
  renta_limpia.csv        → 6,811 filas, 5 columnas
